# Accuracy against known truth

A Monte Carlo simulation knows the true activity in every organ, so it can tell you how far a reconstruction is from the truth, not only how it looks. This tutorial reconstructs a SIMIND simulation of a Lu-177 patient phantom with OSEM and with BSREM, and measures the bias and noise in each organ after every iteration.

You will:

- combine per-organ SIMIND projections into one acquisition with realistic Poisson noise,
- build organ masks from the phantom's ground truth,
- record organ statistics during reconstruction with a callback,
- compare the two algorithms with bias-noise curves.

> **Data.** This tutorial reads `SPECT/SIMIND-MultiOrgan` (SIMIND simulation of a Lu-177 patient phantom, one projection set per organ, with the true activity maps) from the folder set by the `PYTOMOGRAPHY_DATA` environment variable. <a href="../tutorials/data.html#data-spect-simind-multiorgan">Where to get <code>SPECT/SIMIND-MultiOrgan</code></a>.

In [ ]:
import os
from pathlib import Path

# Tutorial data: the folder set by the PYTOMOGRAPHY_DATA environment variable (see Tutorial data in the docs)
DATA = Path(os.environ.get("PYTOMOGRAPHY_DATA", "~/pytomography_data")).expanduser()
# Results go here, never into the data folder
OUTPUT = Path(os.environ.get("PYTOMOGRAPHY_OUTPUT", "pytomography_outputs")).expanduser() / "SPECT/SIMIND-MultiOrgan"
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pytomography
from pytomography.io.SPECT import simind
from pytomography.transforms.SPECT import SPECTAttenuationTransform, SPECTPSFTransform
from pytomography.projectors.SPECT import SPECTSystemMatrix
from pytomography.likelihoods import PoissonLogLikelihood
from pytomography.algorithms import OSEM, BSREM
from pytomography.priors import RelativeDifferencePrior
from pytomography.callbacks import Callback

## 1. A noisy acquisition from per-organ simulations

SIMIND simulated each organ on its own, so the projections of any activity distribution are a weighted sum of the organ projections. SIMIND's projections are counts per MBq per second, so multiplying by each organ's activity and by the time per projection gives the expected counts, and Poisson sampling gives one noisy acquisition.

In [ ]:
path = DATA / 'SPECT' / 'SIMIND-MultiOrgan'
organs = ['bkg', 'liver', 'l_lung', 'r_lung', 'l_kidney', 'r_kidney', 'salivary', 'bladder']
activities = [2500, 450, 7, 7, 100, 100, 20, 90]  # MBq
acquisition_time = 15  # seconds per projection

windows = {w: [str(path / 'multi_projections' / organ / f'{w}.h00') for organ in organs]
           for w in ['photopeak', 'lowerscatter', 'upperscatter']}
object_meta, proj_meta = simind.get_metadata(windows['photopeak'][0])

torch.manual_seed(0)
projections = {w: torch.poisson(simind.combine_projection_data(files, activities) * acquisition_time)
               for w, files in windows.items()}
photopeak = projections['photopeak']

# Scatter in the photopeak window, estimated from the two neighbouring windows (triple energy window)
ww_peak, ww_lower, ww_upper = [simind.get_energy_window_width(windows[w][0]) for w in ['photopeak', 'lowerscatter', 'upperscatter']]
scatter = simind.compute_EW_scatter(projections['lowerscatter'], projections['upperscatter'], ww_lower, ww_upper, ww_peak)
print(f"{tuple(photopeak.shape)} projections, {photopeak.sum().item() / 1e6:.1f} million photopeak counts")

## 2. Organ masks from the ground truth

The phantom's true activity maps are on a finer grid than the reconstruction: 512 × 512 × 768 voxels of 0.075 × 0.075 × 0.15 cm, stored with $z$ first. Averaging blocks of 4 × 4 × 2 voxels gives, for each voxel of the 128 × 128 × 384 reconstruction grid, the fraction the organ fills. A voxel belongs to an organ when the organ fills more than half of it. Each organ's true concentration is its activity divided by its volume.

In [ ]:
truth_shape = (768, 512, 512)            # z, y, x as stored
truth_voxel_mL = 0.075 * 0.075 * 0.15    # cm^3
masks, volume_mL = {}, {}
for organ in organs:
    truth = np.fromfile(path / 'phantom_organs' / f'{organ}_act_av.bin', dtype=np.float32).reshape(truth_shape)
    inside = torch.from_numpy((truth > 0).astype(np.float32)).permute(2, 1, 0)   # to x, y, z
    volume_mL[organ] = inside.sum().item() * truth_voxel_mL
    fraction = F.avg_pool3d(inside[None, None], kernel_size=(4, 4, 2))[0, 0]
    masks[organ] = (fraction > 0.5).to(pytomography.device)
truth_MBq_per_mL = {organ: a / volume_mL[organ] for organ, a in zip(organs, activities)}
for organ in organs:
    print(f"{organ:9s} {volume_mL[organ]:8.0f} mL   {truth_MBq_per_mL[organ]:.4f} MBq/mL   {int(masks[organ].sum()):7d} voxels")

## 3. System matrix and likelihood

The system matrix models attenuation, from the attenuation map SIMIND used, and the collimator-detector response, from the collimator in the SIMIND header. The scatter estimate enters the Poisson likelihood as an additive term.

In [ ]:
attenuation_map = simind.get_attenuation_map(str(path / 'multi_projections' / 'mu208.hct'))
psf_meta = simind.get_psfmeta_from_header(windows['photopeak'][0])
system_matrix = SPECTSystemMatrix(
    obj2obj_transforms=[SPECTAttenuationTransform(attenuation_map), SPECTPSFTransform(psf_meta)],
    proj2proj_transforms=[],
    object_meta=object_meta,
    proj_meta=proj_meta)
likelihood = PoissonLogLikelihood(system_matrix, photopeak, additive_term=scatter)

## 4. Organ statistics during reconstruction

A callback runs after every sub-iteration and returns the object, which the algorithm keeps. This one records, at the end of each iteration, the mean and standard deviation of the voxel values in each organ.

In [ ]:
class OrganStatistics(Callback):
    def __init__(self, masks, n_subsets):
        self.masks = masks
        self.n_subsets = n_subsets
        self.mean = {organ: [] for organ in masks}
        self.std = {organ: [] for organ in masks}

    def run(self, object, n_iter, n_subset):
        if n_subset == self.n_subsets - 1:  # last subset: the end of an iteration
            for organ, mask in self.masks.items():
                values = object[mask]
                self.mean[organ].append(values.mean().item())
                self.std[organ].append(values.std().item())
        return object

## 5. Reconstruct with OSEM and BSREM

BSREM adds a relative difference prior, which penalises differences between neighbouring voxels relative to their values, so it smooths noise without washing out edges as much as a quadratic prior. Both run for 40 iterations of 8 subsets.

In [ ]:
n_iters, n_subsets = 40, 8

t0 = time.time()
stats_osem = OrganStatistics(masks, n_subsets)
recon_osem = OSEM(likelihood)(n_iters=n_iters, n_subsets=n_subsets, callback=stats_osem)
print(f"OSEM:  {time.time() - t0:.1f} s")

t0 = time.time()
stats_bsrem = OrganStatistics(masks, n_subsets)
bsrem = BSREM(likelihood, prior=RelativeDifferencePrior(beta=0.3, gamma=2), relaxation_sequence=lambda n: 1 / (n / 50 + 1))
recon_bsrem = bsrem(n_iters=n_iters, n_subsets=n_subsets, callback=stats_bsrem)
print(f"BSREM: {time.time() - t0:.1f} s")

## 6. From counts to activity

Reconstructed values are counts per voxel. One calibration factor, the total activity divided by the total reconstructed counts, turns them into MBq per voxel, and dividing by the voxel volume gives MBq/mL. On a real scanner the factor comes from imaging a source of known activity.

In [ ]:
voxel_mL = float(np.prod(object_meta.dr))  # SPECT voxel sizes are in cm

def bias_and_noise(stats, recon, organ):
    """Bias and noise of an organ at every iteration, in percent of its true concentration."""
    to_MBq_per_mL = sum(activities) / recon.sum().item() / voxel_mL
    mean = np.array(stats.mean[organ]) * to_MBq_per_mL
    std = np.array(stats.std[organ]) * to_MBq_per_mL
    truth = truth_MBq_per_mL[organ]
    return 100 * (mean / truth - 1), 100 * std / truth

for organ in organs:
    b_osem, n_osem = bias_and_noise(stats_osem, recon_osem, organ)
    b_bsrem, n_bsrem = bias_and_noise(stats_bsrem, recon_bsrem, organ)
    print(f"{organ:9s} after {n_iters} iterations: OSEM bias {b_osem[-1]:+6.1f}% noise {n_osem[-1]:5.1f}%   "
          f"BSREM bias {b_bsrem[-1]:+6.1f}% noise {n_bsrem[-1]:5.1f}%")

## 7. Bias and noise in each organ

Each point is one iteration, starting at the open circle. A perfect reconstruction would sit at zero bias and zero noise.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6), constrained_layout=True)
for ax, organ in zip(axes.ravel(), organs):
    for stats, recon, name, color in [(stats_osem, recon_osem, 'OSEM', 'tab:blue'), (stats_bsrem, recon_bsrem, 'BSREM', 'tab:orange')]:
        bias, noise = bias_and_noise(stats, recon, organ)
        ax.plot(bias, noise, '-', color=color, lw=1.5, label=name)
        ax.plot(bias[0], noise[0], 'o', mfc='none', color=color)
        ax.plot(bias[-1], noise[-1], 'o', color=color)
    ax.axvline(0, color='0.6', lw=0.8)
    ax.set_title(organ)
    ax.set_xlabel('bias (%)')
    ax.set_ylabel('noise (%)')
    ax.grid(alpha=0.3)
axes[0, 0].legend()
plt.show()

In [ ]:
truth_image = sum(masks[organ].float() * truth_MBq_per_mL[organ] for organ in organs).cpu()
images = [truth_image,
          recon_osem.cpu() * sum(activities) / recon_osem.sum().item() / voxel_mL,
          recon_bsrem.cpu() * sum(activities) / recon_bsrem.sum().item() / voxel_mL]
fig, axes = plt.subplots(1, 3, figsize=(9, 7), constrained_layout=True)
for ax, image, title in zip(axes, images, ['Truth', f'OSEM {n_iters}×{n_subsets}', f'BSREM {n_iters}×{n_subsets}']):
    im = ax.imshow(image[:, 64].T, cmap='magma', origin='lower', vmax=0.5, aspect=object_meta.dr[2] / object_meta.dr[0])
    ax.set_title(title)
    ax.axis('off')
fig.colorbar(im, ax=axes, shrink=0.6, label='MBq/mL')
plt.show()

## Next steps

- Shorten `acquisition_time` to see how bias and noise change at low counts.
- Try other priors and values of `beta` in BSREM, or the other algorithms in the [algorithms tutorial](t_algorithms.ipynb).
- Use the same callback with your own phantom: all it needs is a mask and a true concentration for each region.